In [ ]:
# Repository setup for portable, repo-relative paths
from pathlib import Path
import sys

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from dx_chat_entropy.paths import get_paths
PATHS = get_paths(REPO_ROOT)


# Compare LR Estimates

Visualize and compare likelihood ratio estimates across different models.

Two analyses:
1. **KDE overlays** — density plots of LR distributions from each model, on a log scale
2. **Bland-Altman plots** — pairwise agreement between models (multiplicative, x-fold)

## Supporting functions

#### LR Estimate comparisons betwee models

this script takes estimates from different model calls to be compared - doesn't consider the exact predictions (ie not paired differences) but just the overall distributions. 

columsn should be labeled and in a spreadsheet called columns_to_plot.xlsx

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter
from scipy.stats import gaussian_kde
from pathlib import Path

def plot_multi_lr_kde(
    excel_path: str | Path,
    sheet: int | str = 0,
    header_row: int = 0,
    data_start_row: int = 1,
    header_contains: str = r"\bLR\b",   # regex; case-insensitive
    low: float = 1/32,
    high: float = 32,
    n_grid: int = 700,
    bw_method: str | float = "scott",
    min_points: int = 5,
    alpha_bands: float = 0.12,
    legend_loc: str = "upper center",
    legend_ncol: int = 3,
    figsize=(7.0, 4.0),
    savepath: str | Path | None = "columns_to_plot_kde_overlay.pdf",
):
    """
    Overlay KDEs (per log unit) for all columns whose first-row label matches `header_contains`.
    Assumes the data cells in those columns are *LR values* (>0). NaNs & non-positive values are ignored.

    Parameters
    ----------
    excel_path : path to workbook (e.g., 'columns_to_plot.xlsx')
    sheet      : sheet index or name
    header_row : row index containing column labels (default 0)
    data_start_row : first row index containing numeric LR data (default 1)
    header_contains : regex that selects columns by header (default: contains 'LR')
    low, high  : x-range (LR scale), symmetric in log space (default 1/32..32)
    n_grid     : number of x-grid points in log space
    bw_method  : gaussian_kde bandwidth ('scott' | 'silverman' | float)
    min_points : minimum non-NaN values required to draw a KDE for a column
    alpha_bands: alpha for qualitative band shading
    legend_loc : legend location
    legend_ncol: legend columns
    figsize    : figure size
    savepath   : file path to save PDF/PNG; set None to skip saving

    Returns
    -------
    fig, ax
    """
    # -------------------------------------------------------------------------
    # Read sheet exactly as laid out (no header inference)
    df = pd.read_excel(excel_path, sheet_name=sheet, header=None)

    # Identify candidate columns by header regex
    headers = df.iloc[header_row].astype(str)
    mask = headers.str.contains(header_contains, case=True, regex=True)
    cols = np.flatnonzero(mask.values)

    if len(cols) == 0:
        raise ValueError("No columns found whose first-row label matches the regex for 'LR'.")

    # Matplotlib style: clean spines, readable font
    plt.rcParams.update({
        "font.size": 11,
        "axes.spines.right": False,
        "axes.spines.top": False
    })

    # Prebuild x-grid in log10 space; density is per log10 unit (labelled 'per log unit')
    x_log = np.linspace(np.log10(low), np.log10(high), n_grid)
    lr_x  = 10**x_log

    # Qualitative bands (neg/pos strength)
    cuts = [0.1, 0.2, 0.5, 2, 5, 10]
    bands = [(low, 0.1), (0.1, 0.2), (0.2, 0.5), (0.5, 2), (2, 5), (5, 10), (10, high)]
    light = "#fde0dd"; mid = "#fa9fb5"; dark = "#c51b8a"
    band_cols = [dark, mid, light, "#f0f0f0", light, mid, dark]

    # Color cycle for multiple curves
    prop_cycle = plt.rcParams["axes.prop_cycle"]
    colors = prop_cycle.by_key().get("color", ["#0868ac", "#dd3497", "#41ab5d", "#e08214", "#756bb1", "#7b8b8e"])

    fig, ax = plt.subplots(figsize=figsize)

    # Shaded qualitative bands
    for (a, b), col in zip(bands, band_cols):
        ax.axvspan(a, b, color=col, alpha=alpha_bands, lw=0)

    # Iterate columns; compute KDE in log10-space; normalise per log unit
    plotted = 0
    for i, col in enumerate(cols):
        label = str(headers.iloc[col]).strip()

        # Coerce to numeric; use only strictly positive values (LR > 0)
        vals = pd.to_numeric(df.iloc[data_start_row:, col], errors="coerce")
        vals = vals[np.isfinite(vals)]
        vals = vals[vals > 0]

        if vals.size < min_points:
            # Too few points for a stable KDE — skip
            print(f"Skipping '{label}' (n={vals.size} < {min_points}).")
            continue

        log_vals = np.log10(vals.values)
        kde = gaussian_kde(log_vals, bw_method=bw_method)
        pdf = kde(x_log)
        # Normalise so area under pdf over x_log equals 1 (per log unit)
        area = np.trapezoid(pdf, x_log)
        if area > 0:
            pdf = pdf / area

        ax.plot(lr_x, pdf, lw=2.0, label=label, color=colors[i % len(colors)])
        plotted += 1

    if plotted == 0:
        raise ValueError("No columns had enough positive numeric values to plot.")

    # Log-scale x, fractional ticks
    ticks = [1/32, 1/16, 1/8, 1/4, 1/2, 1, 2, 4, 8, 16, 32]
    def frac_label(t: float) -> str:
        return f"1/{int(round(1/t))}" if t < 1 else f"{int(t)}"
    tick_labels = [frac_label(t) for t in ticks]

    ax.set_xscale("log")
    ax.set_xlim(low, high)
    ax.set_xlabel("Likelihood ratio")
    ax.set_ylabel("Density (per log unit)")

    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(FixedFormatter(tick_labels))

    # Reference lines at band cutoffs
    for c in cuts:
        ax.axvline(c, ls="--", lw=0.7, color="grey")

    ax.grid(which="major", ls="--", lw=0.4, alpha=0.5)
    ax.grid(which="minor", lw=0, alpha=0)

    # Legend
    ax.legend(loc=legend_loc, ncol=legend_ncol, frameon=False)

    fig.tight_layout()

    if savepath is not None:
        savepath = Path(savepath)
        fig.savefig(savepath, dpi=600, bbox_inches="tight")
        print(f"Saved figure to {savepath}")

    return fig, ax


# Example call (same folder as your spreadsheet)
fig, ax = plot_multi_lr_kde(REPO_ROOT / 'archive/legacy_runs/lr_estimation_2025_07_21/columns_to_plot.xlsx')

#### Bland Altman Plots for agreement of the same tspreadsheet columns

all the pairwise combinations that start with LR

	•	Reads an Excel sheet.
	•	Selects all columns whose first-row header contains “LR” (regex, case-insensitive).
	•	Detects whether the data are raw LR or log-LR (auto, overrideable).
	•	Generates a separate multiplicative (x-fold) Bland–Altman plot for every pairwise comparison among the selected columns.
	•	Annotates mean bias and 95% LoA in ×-fold units.
	•	Uses shared axis limits across all pairs for comparability.
	•	Saves one PDF per pair under ba_pair_plots/.

In [ ]:
import re
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.transforms import blended_transform_factory

# Optional: Pingouin for built-in BA panel. We fall back to manual if missing.
try:
    import pingouin as pg
    _HAS_PG = True
except Exception:
    _HAS_PG = False


def _sanitize(name: str) -> str:
    """Safe string for filenames."""
    s = re.sub(r"\s+", "_", str(name).strip())
    s = re.sub(r"[^A-Za-z0-9_.-]", "", s)
    return s[:120]


def plot_pairwise_ba_from_excel(
    excel_path: str | Path,
    sheet: int | str = 0,
    header_row: int = 0,
    data_start_row: int = 1,
    header_regex: str = r"LR",        # select columns whose header contains 'LR'
    assume_scale: str = "auto",       # 'auto' | 'lr' | 'ln'
    outdir: str | Path = "ba_pair_plots",
    point_kwargs: dict | None = None, # e.g. {"s": 15, "alpha": 0.7, "color": "#1b8e5a"}
    line_kwargs: dict | None = None,  # e.g. {"color": "black", "lw": 1.1}
    style_science: bool = True,       # try to use SciencePlots if available
) -> list[Path]:
    """
    Create one Bland–Altman plot per pair of selected columns (multiplicative, x-fold).
    Y-axis = ln-ratio (formatted as ×-fold). X-axis = mean ln (formatted as LR).
    """

    # --- style ---------------------------------------------------------------
    if style_science:
        try:
            plt.style.use(["science", "no-latex"])
        except Exception:
            pass
    plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

    # --- read & select columns ----------------------------------------------
    excel_path = Path(excel_path)
    df = pd.read_excel(excel_path, sheet_name=sheet, header=None)

    headers = df.iloc[header_row].astype(str)
    sel = headers[headers.str.contains(header_regex, case=True, regex=True)]
    if sel.empty:
        raise ValueError("No columns matched header_regex; nothing to compare.")

    # Build dict: label -> numeric series
    series_dict: dict[str, pd.Series] = {}
    for col_idx, label in sel.items():
        s = pd.to_numeric(df.iloc[data_start_row:, col_idx], errors="coerce")
        series_dict[str(label).strip()] = s

    labels = list(series_dict.keys())
    if len(labels) < 2:
        raise ValueError("Need ≥2 matching columns to run pairwise BA comparisons.")

    # --- scale detection (LR vs ln-LR) ---------------------------------------
    def _contains_log_hint(names: list[str]) -> bool:
        pat = re.compile(r"\b(ln|log)\b", re.IGNORECASE)
        return any(pat.search(n) for n in names)

    if assume_scale not in {"auto", "lr", "ln"}:
        raise ValueError("assume_scale must be 'auto', 'lr', or 'ln'.")

    if assume_scale == "ln":
        as_log = True
    elif assume_scale == "lr":
        as_log = False
    else:
        # auto: header hint first, else non-positivity
        if _contains_log_hint(labels):
            as_log = True
        else:
            # If any column has any non-positive value, treat as ln (LR cannot be <= 0)
            any_nonpos = any((s.fillna(np.nan) <= 0).any() for s in series_dict.values())
            as_log = any_nonpos

    # --- helper formatters ---------------------------------------------------
    def fmt_exp(x, _):  # display LR values on ln axis
        return f"{np.exp(x):.2g}"
    x_fmt = FuncFormatter(fmt_exp)

    def fmt_fold(x, _):  # display ×-fold on ln-diff axis
        return f"{np.exp(x):.2g}×"
    y_fmt = FuncFormatter(fmt_fold)

    # --- build ln-series and collect global limits ---------------------------
    ln_data = {}
    for lab, s in series_dict.items():
        s = s.astype(float)
        if as_log:
            ln_data[lab] = s.replace([np.inf, -np.inf], np.nan)
        else:
            # raw LR → ln
            s = s.where(s > 0, np.nan)
            ln_data[lab] = np.log(s)

    # Global limits over all pairwise combos
    means_all, diffs_all = [], []
    pairs = list(itertools.combinations(labels, 2))
    for a, b in pairs:
        A, B = ln_data[a], ln_data[b]
        ok = A.notna() & B.notna()
        if ok.sum() < 3:
            continue
        m = (A[ok] + B[ok]) / 2
        d = (A[ok] - B[ok])
        means_all.append(m.values)
        diffs_all.append(d.values)

    if not means_all:
        raise ValueError("No pair had ≥3 overlapping rows after NaN filtering.")

    means_all = np.concatenate(means_all)
    diffs_all = np.concatenate(diffs_all)
    xlim = (means_all.min(), means_all.max())
    ylim = (-1.1 * np.abs(diffs_all).max(), 1.1 * np.abs(diffs_all).max())

    # --- default aesthetics --------------------------------------------------
    if point_kwargs is None:
        point_kwargs = {"s": 18, "alpha": 0.7}
    if line_kwargs is None:
        line_kwargs = {"color": "black", "lw": 1.1}

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    saved: list[Path] = []

    # --- plot each pair ------------------------------------------------------
    for a, b in pairs:
        A, B = ln_data[a], ln_data[b]
        ok = A.notna() & B.notna()
        if ok.sum() < 3:
            print(f"Skip: '{a}' vs '{b}' (n={ok.sum()} < 3).")
            continue

        x = ((A[ok] + B[ok]) / 2).values  # ln geometric mean
        y = (A[ok] - B[ok]).values        # ln ratio A/B

        mu = y.mean()
        sd = y.std(ddof=1)
        loa_hi = mu + 1.96 * sd
        loa_lo = mu - 1.96 * sd

        fig, ax = plt.subplots(figsize=(5.6, 4.0))

        if _HAS_PG:
            # Pingouin BA adds lines; we will re-style them and re-annotate
            pg.plot_blandaltman(A[ok].values, B[ok].values, ax=ax,
                                marker="o", **{k:v for k,v in point_kwargs.items() if k in {"s","alpha","color"}})
            # mean line then LoA lines:
            ax.lines[0].set(**line_kwargs)  # mean
            for ln in ax.lines[1:]:
                ln.set(linestyle="--", **line_kwargs)
        else:
            # Manual BA
            ax.scatter(x, y, marker="o", **point_kwargs)
            ax.axhline(mu, **line_kwargs)
            ax.axhline(loa_hi, linestyle="--", **line_kwargs)
            ax.axhline(loa_lo, linestyle="--", **line_kwargs)

        # remove any auto text (from pg)
        for t in list(ax.texts):
            t.remove()

        # right-side labels in ×-fold
        trans = blended_transform_factory(ax.transAxes, ax.transData)
        ax.text(0.98, mu,     f"Mean: {np.exp(mu):.2f}×",
                transform=trans, ha="right", va="center", fontsize=11)
        ax.text(0.98, loa_hi, f"95% LoA: {np.exp(loa_hi):.2f}×",
                transform=trans, ha="right", va="bottom", fontsize=11)
        ax.text(0.98, loa_lo, f"95% LoA: {np.exp(loa_lo):.2f}×",
                transform=trans, ha="right", va="top", fontsize=11)

        # axes formatting (ln space; label as LR and ×-fold)
        ax.set_xlim(xlim); ax.set_ylim(ylim)
        ax.xaxis.set_major_formatter(x_fmt)
        ax.yaxis.set_major_formatter(y_fmt)

        ax.set_title(f"{a} vs {b}", fontsize=13)
        ax.set_xlabel("Geometric mean LR", fontsize=12)
        ax.set_ylabel(f"LR ratio ({a} / {b})", fontsize=12)

        fig.tight_layout()

        fname = outdir / f"ba_{_sanitize(a)}__vs__{_sanitize(b)}.pdf"
        fig.savefig(fname, format="pdf", bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {fname}")
        saved.append(fname)

    return saved

# Example: columns whose first-row header contains 'LR'
files = plot_pairwise_ba_from_excel(
    REPO_ROOT / 'archive/legacy_runs/lr_estimation_2025_07_21/columns_to_plot.xlsx',
    header_regex=r"LR",        # select headers that contain 'LR'
    assume_scale="auto",       # or 'lr' if raw LRs, 'ln' if log-LRs
    outdir=REPO_ROOT / 'archive/legacy_runs/lr_estimation_2025_07_21/'
)